# K=11 Producao -- execucao LOCAL (Ryzen 7 8-core CPU)

Pipeline de producao do modelo de regressao Bayesiana hierarquica K=11
(10 features baseline + mode_bin) para o produto **Diagnostico de Posicionamento**.

## Estrutura desta pasta

```
scripts/k11_pipeline/
├── 06_k11_pipeline.ipynb     # este notebook
├── train.py                  # treino NUTS K=11
├── evaluate.py               # metricas + asserts
├── export.py                 # exporta JSON para Next.js
└── spotify_tracks_limpo.parquet  # dataset
```

## Pre-requisitos

- **Python 3.10+**
- **CPU com 4+ cores** (testado em Ryzen 7 8-core / 16 threads)
- ~5 GB de espaco em disco para os artefatos
- `pip install -r ../../../requirements.txt` (ou caminho equivalente)

## Como rodar

```bash
cd C:\Users\tito\OneDrive\Documentos\Projetos\spotify_challenge\insights-spotfy-grupo-4\scripts\k11_pipeline
jupyter lab  # ou jupyter notebook
```

Depois clique `Run All` (ou rode celula por celula).

## Arquitetura

- **Target:** `log(popularity + 1)` -> score 0-100 apos `exp() - 1`
- **Features (K=11):** danceability, energy, loudness, speechiness, acousticness, instrumentalness, liveness, valence, tempo, explicit, mode_bin
- **Hierarquica:** 107 generos, intercept + slopes especificos, prior nao-centrado
- **Sampler:** NUTS via NumPyro, 8 chains x 1000 draws, tune=1500, target_accept=0.9
  - **CPU 8-core** (8 chains paralelas, 1 thread cada): **~20-40 min** (Ryzen 7)
  - CPU 4-core (4 chains paralelas): ~50-90 min
  - CPU 2-core (sequential, 1 device): ~70-90 min (Colab free)
- **Validacao:** Train/Val/Test 70/15/15 com SEED=42, asserts RMSE<18, R²>0.30, HDI 0.90-0.97
- **Barra de progresso:** tqdm por chain (ativa em `progressbar=True`)
- **Checkpoint saving:** cada artefato salvo IMEDIATAMENTE apos geracao (sem perder fit)

## Etapas

1. `train.py` -- fit NUTS (~20-40 min no Ryzen 7)
2. `evaluate.py` -- metricas em Val e Test, gera `q11_summary.json`
3. `export.py` -- converte NetCDF em JSON para o backend Next.js

## Configuracao de CPU (editavel em C2)

Variaveis reconhecidas pelo train.py:

- `K11_HOST_DEVICES` (default = `os.cpu_count()`): numero de chains paralelas
- `K11_OMP_THREADS` (default = 1): OpenMP threads por chain
- `K11_PLATFORM` (default = auto): force `cpu` ou `cuda`

Combinacoes para Ryzen 7 (8 cores / 16 threads):

| Config | HOST_DEVICES | OMP_THREADS | Tempo esperado |
|--------|--------------|-------------|----------------|
| Max throughput (recomendado) | 8 | 1 | ~20-40 min |
| Max threads | 4 | 2 | ~30-50 min |
| Conservador (deixa 2 pro OS) | 6 | 1 | ~25-45 min |

Total de threads = HOST_DEVICES x OMP_THREADS. Nao passe do numero de
cores logicos (`os.cpu_count()` = 16 no Ryzen 7).


In [ ]:
# === CPU config (Ryzen 7 8-core) ===
# Estas env vars sao lidas pelo train.py ANTES de importar jax/numpyro.
# Altere aqui e re-rode esta celula para experimentar outras configuracoes.
import os
os.environ['K11_PLATFORM'] = 'cpu'        # forca CPU (sem GPU no seu PC)
os.environ['K11_HOST_DEVICES'] = '8'      # 8 chains paralelas (max throughput)
os.environ['K11_OMP_THREADS'] = '1'       # 1 OMP thread por chain
                                             # 8 chains x 1 thread = 8 threads total
                                             # (deixa 8 threads logicos livres pro OS)

# Para outras combinacoes, descomente uma das opcoes:
# os.environ['K11_HOST_DEVICES'] = '4'    # 4 chains
# os.environ['K11_OMP_THREADS'] = '2'     # 2 threads por chain (= 8 total)

# === Install jax (CPU) + dependencias ===
!pip install -q --upgrade --force-reinstall "jax==0.5.3"
!pip install -q pymc==6.3.1 arviz==1.3.0 pytensor==3.3.0 numpyro==0.21.0 pandas pyarrow scipy scikit-learn

import jax
print()
print('jax version:', jax.__version__)
print('jax devices:', jax.devices())
print('jax backend:', jax.default_backend())
print()
print('K11_HOST_DEVICES =', os.environ.get('K11_HOST_DEVICES', 'auto'))
print('K11_OMP_THREADS  =', os.environ.get('K11_OMP_THREADS', '1'))
print('K11_PLATFORM     =', os.environ.get('K11_PLATFORM', 'auto'))
print('os.cpu_count()   =', os.cpu_count(), '(logical cores do seu CPU)')
print()
if jax.default_backend() == 'cpu':
    n_dev = int(os.environ.get('K11_HOST_DEVICES', os.cpu_count() or 4))
    n_thr = int(os.environ.get('K11_OMP_THREADS', '1'))
    print(f'[OK] JAX em CPU. NUTS levara ~20-40 min com {n_dev} chains x {n_thr} threads.')
else:
    print('[ERRO] JAX em', jax.default_backend(), '-- esperado cpu. Verifique K11_PLATFORM.')
    raise RuntimeError('JAX nao esta em CPU. Confirme K11_PLATFORM=cpu.')



In [ ]:
import os
import sys
from pathlib import Path

# Assume que o notebook esta em scripts/k11_pipeline/.
# Usa o cwd como ponto de partida e sobe niveis se necessario.
PIPELINE_ROOT = Path(os.getcwd())
if not (PIPELINE_ROOT / 'train.py').exists():
    for _ in range(5):
        PIPELINE_ROOT = PIPELINE_ROOT.parent
        if (PIPELINE_ROOT / 'train.py').exists():
            break

os.chdir(PIPELINE_ROOT)
print(f'PIPELINE_ROOT: {PIPELINE_ROOT}')
print()

# Garante diretorios que os scripts escrevem
(PIPELINE_ROOT / 'artifacts').mkdir(parents=True, exist_ok=True)
(PIPELINE_ROOT / 'relatorio' / 'analises' / 'resultados').mkdir(parents=True, exist_ok=True)

# Verificacoes finais
print('=== Estrutura ===')
for f in ['train.py', 'evaluate.py', 'export.py', 'spotify_tracks_limpo.parquet', 'artifacts/', 'relatorio/analises/resultados/']:
    full = PIPELINE_ROOT / f
    status = '[OK]' if full.exists() else '[FALTA]'
    print(f'  {status}  {f}')

assert (PIPELINE_ROOT / 'train.py').exists(), 'train.py nao encontrado -- confira se voce abriu o notebook da pasta k11_pipeline/'
assert (PIPELINE_ROOT / 'evaluate.py').exists(), 'evaluate.py nao encontrado'
assert (PIPELINE_ROOT / 'export.py').exists(), 'export.py nao encontrado'
assert (PIPELINE_ROOT / 'spotify_tracks_limpo.parquet').exists(), 'parquet nao encontrado'
print()
print('Tudo certo. Pode prosseguir para C4 (smoke test).')



In [ ]:
# === SMOKE TEST: valida pipeline end-to-end em ~1-2 min (Ryzen 7) ===
# Usa config minima (10 draws, 10 tune, 2 chains) e escreve em artifacts/smoke/.
# Se esta celula terminar com [OK], o pipeline esta OK para o fit real.
# Se explodir, veja o erro abaixo e NAO rode C5 ate resolver.
import shutil
from pathlib import Path

smoke_dir = Path('artifacts/smoke')
if smoke_dir.exists():
    shutil.rmtree(smoke_dir)
smoke_dir.mkdir(parents=True)

print()
print('=== Smoke test (10/10/2 -> artifacts/smoke/) ===')
print()
!python train.py --draws 10 --tune 10 --chains 2 --out-dir artifacts/smoke 2>&1 | tee smoke.log

smoke_artifacts = Path('artifacts/smoke')
required = ['k11_posterior.nc', 'scaler.json', 'feature_names.json', 'genero_cats.json', 'split_indices.npz']
missing = [f for f in required if not (smoke_artifacts / f).exists()]

print()
if missing:
    print('[ERRO] Smoke test falhou -- faltam:', missing)
    print('NAO rode C5 ate resolver. Veja smoke.log acima.')
    raise SystemExit(1)
else:
    print('[OK] Smoke test passou -- todos os artefatos foram salvos.')
    print('  Posterior:', (smoke_artifacts / 'k11_posterior.nc').stat().st_size // 1024, 'KB')
    print('  Scaler:', (smoke_artifacts / 'scaler.json').stat().st_size, 'bytes')
    print('  Pode prosseguir para C5 (fit real, ~20-40 min no Ryzen 7).')
    shutil.rmtree(smoke_dir)
    if Path('smoke.log').exists():
        Path('smoke.log').unlink()



In [ ]:
# === FIT REAL: K=11 (1000 draws, 1500 tune, 8 chains por padrao) ===
# Configuracao: vem de K11_HOST_DEVICES (8 chains por padrao) + K11_OMP_THREADS (1).
# Tempo esperado no Ryzen 7 (8 cores, 16 threads): ~20-40 min.
# Barra de progresso (tqdm) aparece por chain. ETA calculado ao final.
#
# O posterior (k11_posterior.nc) e salvo IMEDIATAMENTE apos o NUTS,
# antes de qualquer diagnostico. Mesmo se r_hat/ESS ficarem ruins,
# o posterior estara em artifacts/.
!python train.py 2>&1 | tee train.log



In [ ]:
!python evaluate.py 2>&1 | tee evaluate.log



In [ ]:
!python export.py 2>&1 | tee export.log



In [ ]:
import json
from pathlib import Path

print('=== Artefatos gerados ===')
for f in sorted(Path('artifacts').iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:45s}  {size_kb:8.1f} KB')

print()
print('=== Metricas ===')
summary_path = Path('relatorio/analises/resultados/q11_summary.json')
assertions_passed = False
if summary_path.exists():
    with open(summary_path) as fh:
        summary = json.load(fh)
    print(json.dumps(summary, indent=2, ensure_ascii=False))
    # Chave flat (top-level) -- escrita por evaluate.py
    assertions_passed = summary.get('assertions_passed', False)
    print()
    print('[OK] assertions_passed:', assertions_passed)
else:
    print('AVISO: q11_summary.json nao encontrado -- verifique se evaluate.py rodou sem erro.')

if assertions_passed:
    print()
    print('[OK] Pode prosseguir para o backend Next.js (ver C9).')
else:
    print()
    print('[WARN] Assertions falharam. O posterior foi salvo mesmo assim -- pode usar com cautela.')


## Proximos passos

### Se `assertions_passed == true:`

Os artefatos estao em `scripts/k11_pipeline/artifacts/`. Para o backend Next.js:

1. **Copiar artefatos para a raiz do repo** (onde o Next.js espera):

   No PowerShell, na raiz do repo:
   ```powershell
   Copy-Item scripts/k11_pipeline/artifacts/* -Destination artifacts/ -Force
   ```

   Ou no bash:
   ```bash
   cp scripts/k11_pipeline/artifacts/* artifacts/
   ```

2. **Commitar (opcional):**
   ```bash
   git add scripts/k11_pipeline/ artifacts/ scripts/k11_pipeline/*.log
   git commit -m "feat: K=11 modelo treinado e validado"
   git push
   ```

3. **Subir o backend Next.js (na mesma maquina):**
   ```bash
   cd C:\Users\tito\OneDrive\Documentos\Projetos\spotify_challenge\insights-spotfy-grupo-4
   npm install
   Copy-Item .env.local.example .env.local -Force
   # editar .env.local e colocar OPENROUTER_API_KEY=sk-or-v1-...
   npm run dev
   ```

4. **Testar o endpoint:**
   ```bash
   curl -X POST http://localhost:3000/api/diagnose `
     -H "Content-Type: application/json" `
     -d '{"track_features":{"danceability":0.7,"energy":0.5,"loudness":-5.0,"speechiness":0.05,"acousticness":0.3,"instrumentalness":0.0,"liveness":0.1,"valence":0.6,"tempo":120.0,"explicit":0,"mode_bin":0},"genero":"sertanejo"}'
   ```

### Se `assertions_passed == false:`

O posterior foi salvo mesmo assim (veja `artifacts/k11_posterior.nc`). Investigue `q11_summary.json`:

| Metrica | Falha comum | Acao |
|---------|-------------|------|
| RMSE >= 18 | Modelo nao captura variancia | Aumentar K? (Q8 v2 mostrou overfit) |
| R² <= 0.30 | Pouca variancia explicada | Aceitar -- pode ser teto do problema |
| HDI fora de [0.90, 0.97] | Calibracao ruim | Ajustar priors sigma_alpha/sigma_beta |
| r_hat > 1.01 no treino | Chains nao convergiram | Re-rodar com `K11_OMP_THREADS=2` ou `tune=2500` |

## Caveats do modelo

- **Genero deve ser conhecido** -- dropdown tem 107 opcoes apos filtro nao-musical
- **Score e preditivo, nao causal** -- diz o que costuma acontecer, nao como fazer hit
- **Calibrado em popularity do Spotify (0-100)**, nao em qualidade musical
- **NUTS aproxima o posterior** -- HDI e uma estimativa, nao certeza

## Se algo der errado

Logs ficam salvos em:
- `train.log` -- log completo do treino (R-hat, ESS, divergencias, ETA)
- `evaluate.log` -- log da avaliacao
- `export.log` -- log do export
- `artifacts/_error.log` -- traceback se o treino crashou (sempre eh criado em caso de erro)

Para debug, rode os scripts diretamente no terminal (cwd = scripts/k11_pipeline/):
```bash
python train.py
```
e veja o erro com traceback completo.
